In [1]:
# Test DPP sampling success rate on existing traces
import sys
import os
import glob
import pickle
import time
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from loguru import logger
from typing import List, Tuple
from dppy.finite_dpps import FiniteDPP
from transformers import T5EncoderModel, AutoTokenizer
import yaml
from collections import Counter
import re
import matplotlib.pyplot as plt
import seaborn as sns

# Add the project root to the path
sys.path.append("../")
from models.end_to_end.tactic_models.diversity_model.batched_model import BatchedDiversityModel
from utils.diversity_sampling_enhanced import sample_tactics_with_dpp

# Path to the traces
trace_path = "../runs/internlm_dpp/3dprover_internlm_k8/2025_05_13/03_37_51/traces/0/"


In [2]:
# Load configuration files
def load_config(config_path):
    with open(config_path, 'r') as f:
        return yaml.safe_load(f)

diversity_config_path = "../configs/end_to_end/tac_model/diversity_internlm_batched.yaml"
diversity_config = load_config(diversity_config_path)

# Extract the relevant configuration
model_name = diversity_config["diversity_config"]["model"]
max_seq_len = diversity_config["diversity_config"]["max_seq_len"]
num_filtered = diversity_config["diversity_config"]["num_filtered"]
temperature = diversity_config["diversity_config"]["temperature"]
p = diversity_config["diversity_config"]["p"]
ckpt_dir = diversity_config["diversity_config"]["ckpt_dir"]
ckpt_dir = '../' + ckpt_dir if not ckpt_dir.startswith('../') else ckpt_dir
score_network_enabled = diversity_config["diversity_config"].get("score_network", True)
error_weight = diversity_config["diversity_config"].get("error_weight", 1.0)
time_weight = diversity_config["diversity_config"].get("time_weight", 0.0)
error_only = diversity_config["diversity_config"].get("error_only", False)
fixed_size = diversity_config["diversity_config"].get("fixed_size", True)

# Print configuration
print(f"Configuration loaded:")
print(f"- Model: {model_name}")
print(f"- Checkpoint: {ckpt_dir}")
print(f"- Max sequence length: {max_seq_len}")
print(f"- Number of tactics to filter: {num_filtered}")
print(f"- Temperature: {temperature}")
print(f"- p value: {p}")
print(f"- Score network enabled: {score_network_enabled}")
print(f"- Error weight: {error_weight}")
print(f"- Time weight: {time_weight}")
print(f"- Error only: {error_only}")
print(f"- Fixed size: {fixed_size}")


Configuration loaded:
- Model: kaiyuy/leandojo-lean4-tacgen-byt5-small
- Checkpoint: ../runs/internlm_transition_model.ckpt
- Max sequence length: 2300
- Number of tactics to filter: 8
- Temperature: 2
- p value: 0.75
- Score network enabled: True
- Error weight: 0.0
- Time weight: 0.0
- Error only: False
- Fixed size: True


In [3]:
# Load traces
def load_traces(path):
    files = glob.glob(path + '*', recursive=True)
    traces = []
    for file in files:
        try:
            with open(file, 'rb') as f:
                trace = pickle.load(f)
            traces.append(trace)
        except Exception as e:
            print(f"Error loading file {file}: {e}")
    return traces

print(f"Loading traces from {trace_path}")
traces = load_traces(trace_path)
print(f"Loaded {len(traces)} traces")


Loading traces from ../runs/internlm_dpp/3dprover_internlm_k8/2025_05_13/03_37_51/traces/0/
Loaded 245 traces


In [4]:
# Helper class to load the model and run DPP sampling
class DPPTester:
    def __init__(self, config):
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")

        # Create a config object for BatchedDiversityModel
        class ModelConfig:
            def __init__(self, config_dict):
                for key, value in config_dict.items():
                    setattr(self, key, value)

        model_config = ModelConfig({
            "model": model_name,
            "ckpt_dir": ckpt_dir,
            "max_seq_len": max_seq_len,
            "score_network": score_network_enabled,
            "error_weight": error_weight,
            "time_weight": time_weight,
            "error_only": error_only,
            "fixed_size": fixed_size,
            "autoencoder": False
        })

        # Initialize the model
        self.model = BatchedDiversityModel(model_config, self.device)
        self.model.eval()

        print("Model loaded successfully")

    def get_tactic_embeddings(self, tactics, goal, theorem):
        """Get embeddings for tactics given a goal state and theorem."""
        with torch.no_grad():
            encs = []
            tactic_texts = [tac[0] for tac in tactics]

            # Tokenize goal
            tokenized_goal = self.model.tokenizer(
                [theorem + '\n\n' + goal],
                padding="longest",
                max_length=int(self.model.max_seq_len * 1.5),
                truncation=True,
                return_tensors="pt",
            )

            goal_enc = self.model.get_goal_encoding(
                tokenized_goal.input_ids.to(self.device),
                tokenized_goal.attention_mask.to(self.device)
            )

            # Process tactics in chunks
            chunk_size = 64
            for ind in range(0, len(tactic_texts), chunk_size):
                t = tactic_texts[ind:ind + chunk_size]

                tokenized_tactics = self.model.tokenizer(
                    t,
                    padding="longest",
                    max_length=self.model.max_seq_len,
                    truncation=True,
                    return_tensors="pt",
                )

                tactic_ids = tokenized_tactics.input_ids.to(self.device)
                tactic_mask = tokenized_tactics.attention_mask.to(self.device)

                if not self.model.autoencoder:
                    tac_embeds = self.model.tac_encoder.encoder.embed_tokens(tactic_ids)

                    # Set first embedding to be the pooled goal encoding (expanded along batch dimension)
                    tac_embeds_with_goal = torch.cat(
                        [goal_enc.expand(tac_embeds.shape[0], 1, goal_enc.shape[-1]), tac_embeds],
                        dim=1
                    )

                    new_mask = torch.cat(
                        [torch.ones(tactic_mask.shape[0], 1).to(self.device), tactic_mask],
                        dim=1
                    )

                    tac_enc = self.model.tac_encoder(
                        inputs_embeds=tac_embeds_with_goal,
                        attention_mask=new_mask,
                        return_dict=True
                    ).last_hidden_state

                    lens = new_mask.sum(dim=1)
                    tac_enc = (tac_enc * new_mask.unsqueeze(2)).sum(dim=1) / lens.unsqueeze(1)
                    enc = F.normalize(tac_enc, dim=1)

                else:
                    enc = self.model.get_autoencoder_encoding(tactic_ids, tactic_mask)

                encs.append(enc)

            vec_matrix = torch.cat(encs, dim=0)
            return vec_matrix

    def test_dpp_sampling(self, tactics, goal, theorem):
        """Test DPP sampling on tactics.
        
        Returns:
            tuple: (success, sampling_status, error_type)
        """
        if len(tactics) <= 1:
            return True, "success", "single_tactic"  # Always succeeds with 1 or 0 tactics

        try:
            # Get tactic embeddings
            vec_matrix = self.get_tactic_embeddings(tactics, goal, theorem)

            # Generate logprobs/probs
            logprobs = [t[1] / temperature for t in tactics]
            probs = torch.softmax(torch.tensor(logprobs), dim=0)
            probs = probs.to(self.device)

            # Use the enhanced sampling function
            result, _, sampling_status, error_type = sample_tactics_with_dpp(
                tactics=tactics,
                vec_matrix=vec_matrix,
                probs=probs,
                num_filtered=num_filtered,
                p=p,
                score_network=self.model.score_network if self.model.score_network and not self.model.sim_only else None,
                error_weight=self.model.error_weight,
                time_weight=self.model.time_weight,
                error_only=self.model.error_only,
                sim_only=self.model.sim_only,
                fixed_size=self.model.fixed_size,
                device=self.device,
                fallback_strategy="top_k",

            )
            
            # We consider the sampling successful if either:
            # 1. The sampling was successful
            # 2. We had to use a fallback strategy but got results
            success = sampling_status != "error"
                
            return success, sampling_status, error_type
                
        except Exception as e:
            error_msg = str(e)
            if "CUDA out of memory" in error_msg:
                return False, "error", "cuda_oom"
            elif "device-side assert" in error_msg:
                return False, "error", "device_assert"
            elif "tokenizer" in error_msg:
                return False, "error", "tokenizer_error"
            else:
                return False, "error", f"embedding_error:{error_msg[:100]}"


In [5]:
# Initialize the DPP tester
dpp_tester = DPPTester(diversity_config)


Using device: cuda


/home/sean/Documents/bait/models/end_to_end/tactic_models/diversity_model/batched_model.py:54: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(config.ckpt_di

Model loaded successfully


In [6]:
# Define parameter ranges for temperature, scale, error_weight and time_weight to test
temperatures = [1.0, 2.0, 4.0]
scales = [0.1, 1e-2, 5e-2, 0.25, 0.5, 1.0, 5.0, 10.0]
error_weights = [0.0, 0.5, 1.0]
time_weights = [0.0, 0.5, 1.0]

# Results dictionary to store outcomes for different parameter combinations
parameter_results = {}

# Prepare data
data = []
try:
    for trace_idx, trace in enumerate(traces):
        for node_idx, node in tqdm(trace.nodes.items()):
            if node.out_edges and len(node.out_edges) > 0:
                # Extract tactics with logprobs
                tactics = node.data['original_tacs'] if 'original_tacs' in node.data else []

                if tactics:
                    # Get theorem and state from node
                    theorem = node.goal.theorem if hasattr(node.goal, 'theorem') else ""
                    state = node.goal.state if hasattr(node.goal, 'state') else ""

                    data.append((node_idx, (tactics, state, theorem)))

except Exception as e:
    print(f"Error in loading: {e}")


100%|██████████| 2/2 [00:00<00:00, 14614.30it/s]


In [7]:
# Shuffle the data
import random
random.shuffle(data)

# Limit the sample size for parameter exploration
sample_size_per_config = 500  # Smaller sample size per configuration due to increased parameter combinations
data_subset = data[:sample_size_per_config]

# Calculate total number of parameter combinations
total_configs = len(temperatures) * len(scales) * len(error_weights) * len(time_weights)
print(f"Testing {len(temperatures)} temperatures × {len(scales)} scales × {len(error_weights)} error weights × {len(time_weights)} time weights = {total_configs} configurations")
print(f"Running experiments with {len(data_subset)} examples per configuration...")


Testing 3 temperatures × 8 scales × 3 error weights × 3 time weights = 216 configurations
Running experiments with 500 examples per configuration...


In [ ]:
logger.disable(__name__)
# Run experiments with different parameter combinations
for error_weight in error_weights:
    for time_weight in time_weights:
        for temp in temperatures:
            for scale in scales:
                print(f"\nTesting temperature={temp}, scale={scale}, error_weight={error_weight}, time_weight={time_weight}")
                
                # Parameters for this run
                param_key = f"temp_{temp}_scale_{scale}_err_{error_weight}_time_{time_weight}"
                parameter_results[param_key] = {
                    'temperature': temp,
                    'scale': scale,
                    'error_weight': error_weight,
                    'time_weight': time_weight,
                    'total_nodes': 0,
                    'successful_nodes': 0,
                    'fallback_nodes': 0,
                    'failed_nodes': 0,
                    'error_types': Counter(),
                    'sampling_status_counts': Counter(),
                    'results': []
                }
                
                for idx, (node_idx, (tactics, state, theorem)) in tqdm(enumerate(data_subset), total=len(data_subset)):
                    parameter_results[param_key]['total_nodes'] += 1
                    
                    # Test DPP sampling with current parameters
                    with torch.no_grad():
                        # Get tactic embeddings
                        vec_matrix = dpp_tester.get_tactic_embeddings(tactics, state, theorem)
                        
                        # Generate logprobs/probs with current temperature
                        logprobs = [t[1] / temp for t in tactics]
                        probs = torch.softmax(torch.tensor(logprobs), dim=0)
                        probs = probs.to(dpp_tester.device)
                        
                        # Sample using the enhanced sampling function with current parameters
                        try:
                            result, _, sampling_status, error_type = sample_tactics_with_dpp(
                                tactics=tactics,
                                vec_matrix=vec_matrix,
                                probs=probs,
                                num_filtered=num_filtered,
                                p=p,
                                score_network=dpp_tester.model.score_network if dpp_tester.model.score_network else None,
                                error_weight=error_weight,
                                time_weight=time_weight,
                                error_only=dpp_tester.model.error_only,
                                sim_only=False,
                                fixed_size=dpp_tester.model.fixed_size,
                                device=dpp_tester.device,
                                fallback_strategy="top_k",
                                scale=scale
                            )
                            
                            # Track success/fallback status
                            if sampling_status == "success":
                                parameter_results[param_key]['successful_nodes'] += 1
                            elif sampling_status == "fallback":
                                parameter_results[param_key]['fallback_nodes'] += 1
                            else:
                                parameter_results[param_key]['failed_nodes'] += 1
                                
                            # Count error types and sampling status
                            parameter_results[param_key]['error_types'][error_type] += 1
                            parameter_results[param_key]['sampling_status_counts'][sampling_status] += 1
                            
                            # Record the result
                            parameter_results[param_key]['results'].append({
                                'node_idx': node_idx,
                                'num_tactics': len(tactics),
                                'success': sampling_status != "error",
                                'sampling_status': sampling_status,
                                'error_type': error_type
                            })
                            
                        except Exception as e:
                            print(e)
                            # Handle any unexpected errors
                            parameter_results[param_key]['failed_nodes'] += 1
                            parameter_results[param_key]['error_types'][f"unexpected:{str(e)[:50]}"] += 1
                            parameter_results[param_key]['sampling_status_counts']["error"] += 1
                
                # Calculate success rates
                total = parameter_results[param_key]['total_nodes']
                successful = parameter_results[param_key]['successful_nodes']
                fallback = parameter_results[param_key]['fallback_nodes']
                
                pure_success_rate = successful / total if total > 0 else 0
                fallback_rate = fallback / total if total > 0 else 0
                total_success_rate = (successful + fallback) / total if total > 0 else 0
                
                print(f"Results for temperature={temp}, scale={scale}, error_weight={error_weight}, time_weight={time_weight}:")
                print(f"- Pure success rate: {pure_success_rate:.4f} ({successful}/{total})")
                print(f"- Fallback rate: {fallback_rate:.4f} ({fallback}/{total})")
                print(f"- Total success rate: {total_success_rate:.4f} ({successful + fallback}/{total})")



Testing temperature=1.0, scale=0.1, error_weight=0.0, time_weight=0.0


100%|██████████| 500/500 [00:27<00:00, 18.31it/s]


Results for temperature=1.0, scale=0.1, error_weight=0.0, time_weight=0.0:
- Pure success rate: 0.9820 (491/500)
- Fallback rate: 0.0180 (9/500)
- Total success rate: 1.0000 (500/500)

Testing temperature=1.0, scale=0.01, error_weight=0.0, time_weight=0.0


100%|██████████| 500/500 [00:26<00:00, 18.57it/s]


Results for temperature=1.0, scale=0.01, error_weight=0.0, time_weight=0.0:
- Pure success rate: 0.9820 (491/500)
- Fallback rate: 0.0180 (9/500)
- Total success rate: 1.0000 (500/500)

Testing temperature=1.0, scale=0.05, error_weight=0.0, time_weight=0.0


...

100%|██████████| 500/500 [00:26<00:00, 18.59it/s]


Results for temperature=2.0, scale=0.25, error_weight=0.0, time_weight=0.0:
- Pure success rate: 0.9820 (491/500)
- Fallback rate: 0.0180 (9/500)
- Total success rate: 1.0000 (500/500)

Testing temperature=2.0, scale=0.5, error_weight=0.0, time_weight=0.0


100%|██████████| 500/500 [00:26<00:00, 18.57it/s]


Results for temperature=2.0, scale=0.5, error_weight=0.0, time_weight=0.0:
- Pure success rate: 0.9780 (489/500)
- Fallback rate: 0.0220 (11/500)
- Total success rate: 1.0000 (500/500)

Testing temperature=2.0, scale=1.0, error_weight=0.0, time_weight=0.0


  9%|▉         | 45/500 [00:02<00:24, 18.62it/s]

In [ ]:
# Create summary dataframe of results
import pandas as pd

summary_data = []
for param_key, results in parameter_results.items():
    total = results['total_nodes']
    successful = results['successful_nodes']
    fallback = results['fallback_nodes']
    
    pure_success_rate = successful / total if total > 0 else 0
    fallback_rate = fallback / total if total > 0 else 0
    total_success_rate = (successful + fallback) / total if total > 0 else 0
    
    # Find most common error types
    top_errors = []
    for error_type, count in results['error_types'].most_common(3):
        if error_type is not None and error_type != "success":
            top_errors.append(f"{error_type} ({count})")
    
    summary_data.append({
        'Temperature': results['temperature'],
        'Scale': results['scale'],
        'Error_Weight': results['error_weight'],
        'Time_Weight': results['time_weight'],
        'Pure Success Rate': pure_success_rate,
        'Fallback Rate': fallback_rate,
        'Total Success Rate': total_success_rate,
        'Top Errors': ', '.join(top_errors)
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df)


In [ ]:
# Save detailed results to CSV
summary_df.to_csv("dpp_parameter_summary_full.csv", index=False)

# Create a combined dataframe with all individual results
combined_results = []
for param_key, results in parameter_results.items():
    for result in results['results']:
        combined_results.append({
            'Temperature': results['temperature'],
            'Scale': results['scale'],
            'Error_Weight': results['error_weight'],
            'Time_Weight': results['time_weight'],
            'Node ID': result['node_idx'],
            'Num Tactics': result['num_tactics'],
            'Success': result['success'],
            'Sampling Status': result['sampling_status'],
            'Error Type': result['error_type']
        })

combined_df = pd.DataFrame(combined_results)
combined_df.to_csv("dpp_parameter_detailed_results_full.csv", index=False)


In [ ]:
# Create visualizations for impact of all parameters
# We'll need to use slices of the data to create 2D heatmaps

# Best parameters based on pure success rate
best_row = summary_df.loc[summary_df["Pure Success Rate"].idxmax()]
best_temp = best_row['Temperature']
best_scale = best_row['Scale'] 
best_error_weight = best_row['Error_Weight']
best_time_weight = best_row['Time_Weight']

print("\nBest overall parameter combination:")
print(f"Temperature = {best_temp}, Scale = {best_scale}, Error Weight = {best_error_weight}, Time Weight = {best_time_weight}")
print(f"Pure Success Rate: {best_row['Pure Success Rate']:.4f}")
print(f"Total Success Rate: {best_row['Total Success Rate']:.4f}")


In [ ]:
# Create heatmap of temperature vs scale (at best error_weight and time_weight)
plt.figure(figsize=(10, 8))
filtered_df = summary_df[(summary_df['Error_Weight'] == best_error_weight) & 
                         (summary_df['Time_Weight'] == best_time_weight)]

heatmap_temp_scale = filtered_df.pivot(index="Temperature", columns="Scale", values="Pure Success Rate")
sns.heatmap(heatmap_temp_scale, annot=True, cmap="YlGnBu", fmt=".3f", vmin=0, vmax=1)
plt.title(f'Pure Success Rate by Temperature and Scale\n(Error Weight={best_error_weight}, Time Weight={best_time_weight})')
plt.tight_layout()
plt.savefig('dpp_success_heatmap_temp_scale.png')
plt.show()


In [ ]:
# Create heatmap of error_weight vs time_weight (at best temperature and scale)
plt.figure(figsize=(10, 8))
filtered_df = summary_df[(summary_df['Temperature'] == best_temp) & 
                         (summary_df['Scale'] == best_scale)]

heatmap_error_time = filtered_df.pivot(index="Error_Weight", columns="Time_Weight", values="Pure Success Rate")
sns.heatmap(heatmap_error_time, annot=True, cmap="YlGnBu", fmt=".3f", vmin=0, vmax=1)
plt.title(f'Pure Success Rate by Error Weight and Time Weight\n(Temperature={best_temp}, Scale={best_scale})')
plt.tight_layout()
plt.savefig('dpp_success_heatmap_error_time.png')
plt.show()


In [ ]:
# Create heatmap of error_weight vs temperature (at best scale and time_weight)
plt.figure(figsize=(10, 8))
filtered_df = summary_df[(summary_df['Scale'] == best_scale) & 
                         (summary_df['Time_Weight'] == best_time_weight)]

heatmap_error_temp = filtered_df.pivot(index="Error_Weight", columns="Temperature", values="Pure Success Rate")
sns.heatmap(heatmap_error_temp, annot=True, cmap="YlGnBu", fmt=".3f", vmin=0, vmax=1)
plt.title(f'Pure Success Rate by Error Weight and Temperature\n(Scale={best_scale}, Time Weight={best_time_weight})')
plt.tight_layout()
plt.savefig('dpp_success_heatmap_error_temp.png')
plt.show()


In [ ]:
# Create heatmap of scale vs error_weight (at best temperature and time_weight)
plt.figure(figsize=(10, 8))
filtered_df = summary_df[(summary_df['Temperature'] == best_temp) & 
                         (summary_df['Time_Weight'] == best_time_weight)]

heatmap_scale_error = filtered_df.pivot(index="Scale", columns="Error_Weight", values="Pure Success Rate")
sns.heatmap(heatmap_scale_error, annot=True, cmap="YlGnBu", fmt=".3f", vmin=0, vmax=1)
plt.title(f'Pure Success Rate by Scale and Error Weight\n(Temperature={best_temp}, Time Weight={best_time_weight})')
plt.tight_layout()
plt.savefig('dpp_success_heatmap_scale_error.png')
plt.show()


In [ ]:
# Analyze effect of each parameter independently
param_effects = {
    'Temperature': [],
    'Scale': [],
    'Error_Weight': [],
    'Time_Weight': []
}

# Temperature effect (averaged across other parameters)
for temp in temperatures:
    temp_df = summary_df[summary_df['Temperature'] == temp]
    avg_success = temp_df['Pure Success Rate'].mean()
    param_effects['Temperature'].append((temp, avg_success))

# Scale effect
for scale in scales:
    scale_df = summary_df[summary_df['Scale'] == scale]
    avg_success = scale_df['Pure Success Rate'].mean()
    param_effects['Scale'].append((scale, avg_success))

# Error weight effect
for ew in error_weights:
    ew_df = summary_df[summary_df['Error_Weight'] == ew]
    avg_success = ew_df['Pure Success Rate'].mean()
    param_effects['Error_Weight'].append((ew, avg_success))

# Time weight effect
for tw in time_weights:
    tw_df = summary_df[summary_df['Time_Weight'] == tw]
    avg_success = tw_df['Pure Success Rate'].mean()
    param_effects['Time_Weight'].append((tw, avg_success))

# Plot the effect of each parameter
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Effect of Individual Parameters on Pure Success Rate', fontsize=16)

for i, (param, values) in enumerate(param_effects.items()):
    row, col = i // 2, i % 2
    x_vals = [v[0] for v in values]
    y_vals = [v[1] for v in values]
    
    axes[row, col].plot(x_vals, y_vals, 'o-')
    axes[row, col].set_title(f'Effect of {param}')
    axes[row, col].set_xlabel(param)
    axes[row, col].set_ylabel('Average Pure Success Rate')
    axes[row, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.savefig('parameter_effects.png')
plt.show()


In [ ]:
# Plot error types by parameter combination
# Group less frequent error types
error_threshold = 3
filtered_errors = combined_df.copy()
error_counts = filtered_errors['Error Type'].value_counts()
filtered_errors.loc[~filtered_errors['Error Type'].isin(error_counts[error_counts >= error_threshold].index), 'Error Type'] = 'other'

# Plot error counts across all parameters
plt.figure(figsize=(14, 8))
error_by_param_combo = pd.crosstab(
    [filtered_errors['Temperature'], filtered_errors['Scale'], 
     filtered_errors['Error_Weight'], filtered_errors['Time_Weight']], 
    filtered_errors['Error Type']
)

error_totals = error_by_param_combo.sum(axis=1).sort_values(ascending=False)
top_configs = error_totals.index[:10]  # Top 10 parameter combinations by error count
error_by_top_configs = error_by_param_combo.loc[top_configs]

# Create labels for parameter combinations
config_labels = [f"T:{t},S:{s},E:{e},TW:{tw}" for t, s, e, tw in top_configs]
error_by_top_configs.index = config_labels

error_by_top_configs.plot(kind='bar', stacked=True, figsize=(15, 8))
plt.title('Error Types by Top Parameter Combinations')
plt.ylabel('Count')
plt.xlabel('Parameter Combination (Temp, Scale, Error Weight, Time Weight)')
plt.xticks(rotation=45)
plt.legend(title='Error Type', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('error_by_param_combo.png')
plt.show()


In [ ]:
# Create 3D plot for temperature, scale, and error_weight at best time_weight
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

filtered_df = summary_df[summary_df['Time_Weight'] == best_time_weight]
for ew in error_weights:
    df = filtered_df[filtered_df['Error_Weight'] == ew]
    temps = df['Temperature'].values
    scales = df['Scale'].values
    success = df['Pure Success Rate'].values
    ax.scatter(temps, scales, success, label=f'Error Weight={ew}', s=100)

ax.set_xlabel('Temperature')
ax.set_ylabel('Scale')
ax.set_zlabel('Pure Success Rate')
ax.set_title(f'DPP Success Rate by Temperature, Scale, and Error Weight (Time Weight={best_time_weight})')
ax.legend()
plt.tight_layout()
plt.savefig('dpp_3d_plot.png')
plt.show()


In [ ]:
# Final recommendations based on analysis
print("\n===== Final Recommendations =====")
print("Based on the comprehensive parameter analysis:")
print(f"Optimal parameter configuration:")
print(f"- Temperature: {best_temp}")
print(f"- Scale: {best_scale}")
print(f"- Error Weight: {best_error_weight}")
print(f"- Time Weight: {best_time_weight}")
print(f"\nRecommendation for diversity_internlm_batched.yaml config:")
print("```yaml")
print("diversity_config:")
print(f"  model: kaiyuy/leandojo-lean4-tacgen-byt5-small")
print(f"  max_seq_len: 2300")
print(f"  num_filtered: {num_filtered}")
print(f"  temperature: {best_temp}")
print(f"  p: {p}")
print(f"  scale: {best_scale}")
print(f"  score_network: {best_error_weight > 0 or best_time_weight > 0}")
print(f"  error_weight: {best_error_weight}")
print(f"  time_weight: {best_time_weight}")
print(f"  error_only: {error_only}")
print(f"  fixed_size: {fixed_size}")
print("```")
